# SUPERVISED LEARNING FOR CLUSTERING ANALYSIS
## Complete Analysis: Metrics, Hyperparameter Tuning, Feature Importance & Cluster Interpretation

**Objective**: Evaluate the predictive power of customer features and analyze their importance for characterizing customer clusters using supervised learning methods.

---

## PART 1: MULTI-CLASS CLASSIFICATION METRICS

### 1.1 Introduction to Multi-class Classification

This is a multi-class classification problem where the target variable has more than 2 values (customer clusters).

**Important metrics for multi-class classification:**
1. **Accuracy** - Overall correctness rate
2. **Precision, Recall, F1-Score** (macro, weighted) - Per-class accuracy
3. **Balanced Accuracy** - Average recall across classes
4. **Cohen's Kappa** - Agreement beyond random chance
5. **Matthews Correlation Coefficient (MCC)** - Comprehensive correlation coefficient
6. **Confusion Matrix** - Error breakdown table
7. **Jaccard Score** - Similarity coefficient

---

### 1.2 Detailed Metrics Explanation

#### 1.2.1 **ACCURACY**

**Formula:**
$$\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN} = \frac{\text{Correct Predictions}}{\text{Total Predictions}}$$

**Meaning:**
- Overall percentage of correct predictions
- Most basic metric

**When to use:**
- Balanced data (similar size classes)
- All mistakes are equally important

**Limitations:**
- Heavily biased with imbalanced data
- Does not show per-class performance

---

#### 1.2.2 **PRECISION**

**Formula (for class i):**
$$\text{Precision}_i = \frac{TP_i}{TP_i + FP_i}$$

**Meaning:**
- Of all predicted as class i, how many are actually class i?
- Measures accuracy of positive predictions

**When to use:**
- False positives are expensive
- Example: Spam detection - don't want to block important emails
- Example: Medical diagnosis - don't want false alarms

---

#### 1.2.3 **RECALL** (Also called Sensitivity)

**Formula (for class i):**
$$\text{Recall}_i = \frac{TP_i}{TP_i + FN_i}$$

**Meaning:**
- Of all actually class i, how many did we find?
- Measures ability to catch positive cases

**When to use:**
- False negatives are expensive
- Example: Disease detection - don't want to miss patients
- Example: Fraud detection - don't want to miss fraudulent transactions

---

#### 1.2.4 **F1-SCORE** (Harmonic Mean)

**Formula (for class i):**
$$F1_i = 2 \times \frac{\text{Precision}_i \times \text{Recall}_i}{\text{Precision}_i + \text{Recall}_i}$$

**Meaning:**
- Balance between Precision and Recall
- Harmonic mean gives equal weight to both

**When to use:**
- Imbalanced data
- Need balance between Precision and Recall
- The "golden" metric for imbalanced multi-class problems

**Two F1-Score types:**
- **Macro F1**: Simple average of F1 for each class - treats all classes equally
- **Weighted F1**: Weighted average by class size - accounts for imbalance

---

#### 1.2.5 **BALANCED ACCURACY**

**Formula:**
$$\text{Balanced Accuracy} = \frac{1}{n} \sum_{i=1}^{n} \text{Recall}_i$$

**Meaning:**
- Average recall across all classes
- Balances accuracy for imbalanced data

**When to use:**
- Imbalanced data
- Want equal performance on all classes

---

#### 1.2.6 **COHEN'S KAPPA**

**Formula:**
$$\kappa = \frac{p_o - p_e}{1 - p_e}$$

Where:
- $p_o$ = Observed accuracy
- $p_e$ = Expected accuracy by chance

**Meaning:**
- Agreement beyond random chance
- Adjusts for baseline class probabilities

**When to use:**
- Imbalanced data
- Want fair comparison between models
- Excellent for annotator agreement

**Kappa Interpretation:**
- $\kappa > 0.9$: Almost perfect agreement
- $0.8 < \kappa \leq 0.9$: Very good
- $0.6 < \kappa \leq 0.8$: Good
- $0.4 < \kappa \leq 0.6$: Moderate
- $\kappa \leq 0.4$: Poor

---

#### 1.2.7 **MATTHEWS CORRELATION COEFFICIENT (MCC)**

**Formula (binary):**
$$MCC = \frac{TP \times TN - FP \times FN}{\sqrt{(TP+FP)(TP+FN)(TN+FP)(TN+FN)}}$$

**Meaning:**
- Matthews correlation coefficient
- Considers all 4 confusion matrix components
- Not biased by class imbalance

**When to use:**
- Imbalanced data
- Want most balanced and fair metric
- Excellent for imbalanced datasets

**Advantages:**
- Better than Accuracy for imbalanced data
- Not affected by class ratio
- Range [-1, 1], easy to interpret

---

#### 1.2.8 **CONFUSION MATRIX**

**Definition:** 2x2 table (or nxn for multi-class) showing:
- Rows: Actual labels
- Columns: Predicted labels

**Components:**
- **TP (True Positive)**: Correctly predicted as positive
- **TN (True Negative)**: Correctly predicted as negative
- **FP (False Positive)**: Wrongly predicted as positive (false alarm)
- **FN (False Negative)**: Wrongly predicted as negative (miss)

**When to use:**
- Detailed error analysis
- Understand which types of errors model makes
- Basis for calculating other metrics

---

#### 1.2.9 **JACCARD SCORE**

**Formula:**
$$\text{Jaccard} = \frac{TP}{TP + FP + FN}$$

**Meaning:**
- Ratio of intersection to union
- Also called Jaccard Similarity

**When to use:**
- Multi-class classification
- Sensitive to both false positives and false negatives
- Good for object detection problems

---

### 1.3 Metrics Comparison Table

In [ ]:
import pandas as pd
import sys
sys.path.insert(0, '..')

from src.metrics import MetricsCalculator

# Create metrics comparison table
metrics_table = MetricsCalculator.get_metrics_definition_table()
print("\n" + "="*150)
print("METRICS COMPARISON TABLE FOR MULTI-CLASS CLASSIFICATION")
print("="*150)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)
print(metrics_table.to_string(index=False))

### 1.4 Recommendations for Choosing Metrics

**Situation 1: Balanced data**
- Use: **Accuracy**, **Precision/Recall/F1-Score (macro)**

**Situation 2: Imbalanced data**
- Use: **Balanced Accuracy**, **F1-Score (weighted)**, **Cohen's Kappa**, **MCC**
- Avoid: Accuracy (biased by largest class)

**Situation 3: Different error costs**
- High False Positives: Optimize for **Precision**
- High False Negatives: Optimize for **Recall**
- Equal cost: Optimize for **F1-Score**

**Situation 4: Want most comprehensive metric**
- Use: **Matthews Correlation Coefficient (MCC)**

---

## PART 2: HYPERPARAMETER TUNING WITH MULTIPLE MODELS

### 2.1 Load Data and Preprocessing

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

from src.preprocessing import DataPreprocessor
from src.metrics import MetricsCalculator
from src.model_utils import ModelTrainer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "dataset").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "customer_clusters_with_features.csv"

# Load data
df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nData types:")
print(df.dtypes)
print("\nTarget variable (Cluster) distribution:")
print(df['Cluster'].value_counts().sort_index())

In [ ]:
# Preprocessing
preprocessor = DataPreprocessor(random_state=42)

# Separate features and target
TARGET_COL = 'Cluster'
DROP_COLS = []  # Adjust if needed

X = df.drop(columns=[TARGET_COL] + DROP_COLS)
y = df[TARGET_COL]

print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Classes: {y.nunique()}")
print(f"\nClass distribution:\n{y.value_counts().sort_index()}")

# Get feature types
numeric_features, categorical_features = preprocessor.get_feature_types(df, TARGET_COL, DROP_COLS)
print(f"\nNumeric features: {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")

# Build preprocessor
col_transformer = preprocessor.build_preprocessor(numeric_features, categorical_features)

# Split data
X_train, X_test, y_train, y_test = preprocessor.split_data(X, y, test_size=0.2, stratify=True)
print(f"\nTrain set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

### 2.2 Model Training with Hyperparameter Tuning

In [ ]:
from sklearn.pipeline import Pipeline

# Create model trainer
trainer = ModelTrainer(random_state=42)

# Models to train
model_names = ['logistic', 'random_forest', 'gradient_boosting', 'svm', 'knn', 'decision_tree']

print("\n" + "="*100)
print("HYPERPARAMETER TUNING FOR ALL MODELS")
print("="*100 + "\n")

best_models_dict = {}
cv_results_dict = {}

for model_name in model_names:
    print(f"\n{'='*50}")
    print(f"Training: {model_name.upper()}")
    print(f"{'='*50}")
    
    # Create pipeline
    pipeline = trainer.create_pipeline(model_name)
    
    # Grid Search CV
    print(f"Performing Grid Search...")
    gs = trainer.grid_search_cv(pipeline, X_train, y_train, cv=5, n_jobs=-1, verbose=1)
    
    print(f"\nBest parameters: {gs.best_params_}")
    print(f"Best CV score: {gs.best_score_:.4f}")
    
    best_models_dict[model_name] = gs.best_estimator_
    
    # Cross-validation on best model
    cv_results = trainer.cross_validate_model(gs.best_estimator_, X_train, y_train, cv=5)
    cv_results_dict[model_name] = cv_results
    
    print(f"\nCV Results:")
    print(f"  Accuracy: {cv_results['test_accuracy'].mean():.4f} (+/- {cv_results['test_accuracy'].std():.4f})")
    print(f"  F1 (weighted): {cv_results['test_f1_weighted'].mean():.4f} (+/- {cv_results['test_f1_weighted'].std():.4f})")
    print(f"  F1 (macro): {cv_results['test_f1_macro'].mean():.4f} (+/- {cv_results['test_f1_macro'].std():.4f})")

### 2.3 Model Comparison - Test Set Performance

In [ ]:
# Test set performance
test_results = {}

for model_name, model in best_models_dict.items():
    y_pred = model.predict(X_test)
    
    metrics_calc = MetricsCalculator()
    metrics = metrics_calc.calculate_all_metrics(y_test, y_pred)
    
    test_results[model_name] = {
        'accuracy': metrics['accuracy'],
        'balanced_accuracy': metrics['balanced_accuracy'],
        'f1_macro': metrics['f1_macro'],
        'f1_weighted': metrics['f1_weighted'],
        'precision_macro': metrics['precision_macro'],
        'recall_macro': metrics['recall_macro'],
        'cohen_kappa': metrics['cohen_kappa'],
        'matthews_corrcoef': metrics['matthews_corrcoef'],
    }

# Create comparison dataframe
comparison_df = pd.DataFrame(test_results).T

print("\n" + "="*100)
print("TEST SET PERFORMANCE COMPARISON")
print("="*100)
print(comparison_df.round(4))
print("\n" + "="*100)

### 2.4 Visualization - Model Comparison

In [ ]:
# Visualization
fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')

metrics_to_plot = ['accuracy', 'f1_weighted', 'cohen_kappa', 'matthews_corrcoef']

for idx, (ax, metric) in enumerate(zip(axes.flat, metrics_to_plot)):
    comparison_df[metric].sort_values().plot(kind='barh', ax=ax, color='steelblue')
    ax.set_title(f'{metric.replace("_", " ").title()}')
    ax.set_xlabel('Score')
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: results/model_comparison.png")

### 2.5 Overfitting Analysis

In [ ]:
# Check overfitting - Train vs Test performance
overfitting_analysis = {}

for model_name, model in best_models_dict.items():
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    train_acc = (y_train_pred == y_train).mean()
    test_acc = (y_test_pred == y_test).mean()
    gap = train_acc - test_acc
    
    overfitting_analysis[model_name] = {
        'train_accuracy': train_acc,
        'test_accuracy': test_acc,
        'gap': gap,
        'status': 'OVERFITTING' if gap > 0.1 else 'OK' if gap > 0.02 else 'UNDERFITTING'
    }

overfitting_df = pd.DataFrame(overfitting_analysis).T
print("\n" + "="*100)
print("OVERFITTING ANALYSIS (Train vs Test)")
print("="*100)
print(overfitting_df.round(4))

# Visualization
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(overfitting_df))
width = 0.35

ax.bar(x - width/2, overfitting_df['train_accuracy'], width, label='Train', color='green', alpha=0.7)
ax.bar(x + width/2, overfitting_df['test_accuracy'], width, label='Test', color='red', alpha=0.7)
ax.axhline(y=0.9, color='gray', linestyle='--', linewidth=1, label='Target')

ax.set_xlabel('Model')
ax.set_ylabel('Accuracy')
ax.set_title('Train vs Test Accuracy - Overfitting Check')
ax.set_xticks(x)
ax.set_xticklabels(overfitting_df.index, rotation=45)
ax.legend()
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/overfitting_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: results/overfitting_analysis.png")

## PART 4: CLUSTER ANALYSIS WITH IMPORTANT FEATURES

### 4.1 Analyze Important Features by Cluster

In [ ]:
# Get feature names after preprocessing
feature_names_numeric = numeric_features
if categorical_features:
    # After OneHotEncoding
    cat_features_encoded = col_transformer.named_transformers_['cat'].get_feature_names_out(categorical_features).tolist()
    all_features = feature_names_numeric + cat_features_encoded
else:
    all_features = feature_names_numeric

print(f"Total features after preprocessing: {len(all_features)}")
print(f"Features: {all_features[:20]}...")  # Print first 20

In [ ]:
# Extract feature importances from tree-based models
feature_importance_dict = {}

for model_name in ['random_forest', 'gradient_boosting', 'decision_tree']:
    model = best_models_dict[model_name]
    
    # Get the estimator from pipeline
    estimator = model.named_steps[model_name]
    
    importances = estimator.feature_importances_
    
    importance_df = pd.DataFrame({
        'feature': all_features,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    feature_importance_dict[model_name] = importance_df
    
    print(f"\n{model_name.upper()} - Top 15 Features:")
    print(importance_df.head(15).to_string(index=False))

In [ ]:
# Extract coefficients from Logistic Regression
lr_model = best_models_dict['logistic']
lr_estimator = lr_model.named_steps['logistic']

# For multi-class, coefficients shape is (n_classes, n_features)
# Take mean absolute coefficient value
coef_abs_mean = np.abs(lr_estimator.coef_).mean(axis=0)

lr_importance_df = pd.DataFrame({
    'feature': all_features,
    'importance': coef_abs_mean
}).sort_values('importance', ascending=False)

print("\nLOGISTIC REGRESSION - Top 15 Features (by |coefficient|):")
print(lr_importance_df.head(15).to_string(index=False))

### 3.2 Consensus Feature Importance

In [ ]:
# Combine importances from all models (normalize first)
importances_combined = {}

for feature in all_features:
    importances_combined[feature] = []

# Random Forest
rf_norm = feature_importance_dict['random_forest'].set_index('feature')['importance']
rf_norm = rf_norm / rf_norm.sum()
for feat, val in rf_norm.items():
    importances_combined[feat].append(val)

# Gradient Boosting
gb_norm = feature_importance_dict['gradient_boosting'].set_index('feature')['importance']
gb_norm = gb_norm / gb_norm.sum()
for feat, val in gb_norm.items():
    importances_combined[feat].append(val)

# Decision Tree
dt_norm = feature_importance_dict['decision_tree'].set_index('feature')['importance']
dt_norm = dt_norm / dt_norm.sum()
for feat, val in dt_norm.items():
    importances_combined[feat].append(val)

# Logistic Regression
lr_norm = lr_importance_df.set_index('feature')['importance']
lr_norm = lr_norm / lr_norm.sum()
for feat, val in lr_norm.items():
    importances_combined[feat].append(val)

# Calculate mean importance
consensus_importance = pd.DataFrame({
    'feature': list(importances_combined.keys()),
    'mean_importance': [np.mean(v) for v in importances_combined.values()],
    'std_importance': [np.std(v) for v in importances_combined.values()]
}).sort_values('mean_importance', ascending=False)

print("\n" + "="*80)
print("CONSENSUS FEATURE IMPORTANCE (Average across all models)")
print("="*80)
print(consensus_importance.head(20).to_string(index=False))

### 3.3 Visualization - Feature Importance

In [ ]:
# Plot top features from each model
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Feature Importance Across Models', fontsize=16, fontweight='bold')

top_n = 10

# Random Forest
rf_top = feature_importance_dict['random_forest'].head(top_n)
axes[0, 0].barh(rf_top['feature'], rf_top['importance'], color='green')
axes[0, 0].set_title('Random Forest')
axes[0, 0].invert_yaxis()

# Gradient Boosting
gb_top = feature_importance_dict['gradient_boosting'].head(top_n)
axes[0, 1].barh(gb_top['feature'], gb_top['importance'], color='blue')
axes[0, 1].set_title('Gradient Boosting')
axes[0, 1].invert_yaxis()

# Decision Tree
dt_top = feature_importance_dict['decision_tree'].head(top_n)
axes[1, 0].barh(dt_top['feature'], dt_top['importance'], color='orange')
axes[1, 0].set_title('Decision Tree')
axes[1, 0].invert_yaxis()

# Consensus
consensus_top = consensus_importance.head(top_n)
axes[1, 1].barh(consensus_top['feature'], consensus_top['mean_importance'], color='red', xerr=consensus_top['std_importance'])
axes[1, 1].set_title('Consensus (Average ± Std)')
axes[1, 1].invert_yaxis()

for ax in axes.flat:
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('results/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: results/feature_importance.png")

### 3.4 Retrain Model with Top Features Only

In [ ]:
# Select top N features
top_features_list = consensus_importance.head(10)['feature'].tolist()
print(f"Top 10 features: {top_features_list}")

# Need to map back to original feature names
# This is complex due to preprocessing, so let's use feature selection approach instead

from sklearn.feature_selection import SelectKBest, f_classif

# Apply feature selection on transformed data
X_train_transformed = col_transformer.fit_transform(X_train)
X_test_transformed = col_transformer.transform(X_test)

# Select top 10 features
selector = SelectKBest(f_classif, k=10)
X_train_selected = selector.fit_transform(X_train_transformed, y_train)
X_test_selected = selector.transform(X_test_transformed)

selected_feature_indices = selector.get_support(indices=True)
selected_features_names = [all_features[i] for i in selected_feature_indices]

print(f"\nSelected {len(selected_features_names)} features:")
for i, feat in enumerate(selected_features_names, 1):
    print(f"  {i}. {feat}")

In [ ]:
# Retrain best model with selected features only
best_model_name = comparison_df['f1_weighted'].idxmax()
print(f"Best model: {best_model_name}")

# Train on full features
model_full = best_models_dict[best_model_name]
y_pred_full = model_full.predict(X_test)

# Train on selected features
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

if best_model_name == 'random_forest':
    model_selected = RandomForestClassifier(
        n_estimators=best_models_dict[best_model_name].named_steps[best_model_name].n_estimators,
        max_depth=best_models_dict[best_model_name].named_steps[best_model_name].max_depth,
        random_state=42
    )
else:
    model_selected = LogisticRegression(max_iter=5000, random_state=42, multi_class='multinomial')

model_selected.fit(X_train_selected, y_train)
y_pred_selected = model_selected.predict(X_test_selected)

# Compare results
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score

metrics_comparison = pd.DataFrame({
    'Metric': ['Accuracy', 'F1-Score (weighted)', 'F1-Score (macro)', 'Balanced Accuracy'],
    'Full Features': [
        accuracy_score(y_test, y_pred_full),
        f1_score(y_test, y_pred_full, average='weighted'),
        f1_score(y_test, y_pred_full, average='macro'),
        balanced_accuracy_score(y_test, y_pred_full)
    ],
    'Top 10 Features': [
        accuracy_score(y_test, y_pred_selected),
        f1_score(y_test, y_pred_selected, average='weighted'),
        f1_score(y_test, y_pred_selected, average='macro'),
        balanced_accuracy_score(y_test, y_pred_selected)
    ]
})

metrics_comparison['Difference'] = metrics_comparison['Top 10 Features'] - metrics_comparison['Full Features']

print("\n" + "="*80)
print("RETRAINED MODEL: FULL FEATURES vs TOP 10 FEATURES")
print("="*80)
print(metrics_comparison.to_string(index=False))

## PART 4: CLUSTER ANALYSIS WITH IMPORTANT FEATURES

### 4.1 Analyze Important Features by Cluster

In [ ]:
# Add predictions to original data for analysis
df_with_pred = df.copy()
df_with_pred['predicted_cluster'] = model_full.predict(X)

# Analyze important features by cluster
print("\n" + "="*80)
print("CLUSTER ANALYSIS - IMPORTANT FEATURES STATISTICS")
print("="*80 + "\n")

for cluster_id in sorted(df['Cluster'].unique()):
    cluster_mask = df['Cluster'] == cluster_id
    cluster_data = df[cluster_mask]
    
    print(f"\n{'='*60}")
    print(f"CLUSTER {cluster_id} (n={len(cluster_data)})")
    print(f"{'='*60}")
    
    # Statistics for top features
    for feat in selected_features_names[:5]:
        if feat in numeric_features:
            mean_val = cluster_data[feat].mean()
            std_val = cluster_data[feat].std()
            min_val = cluster_data[feat].min()
            max_val = cluster_data[feat].max()
            print(f"\n  {feat}:")
            print(f"    Mean: {mean_val:.2f} ± {std_val:.2f} [min: {min_val:.2f}, max: {max_val:.2f}]")
        else:
            print(f"\n  {feat}:")
            print(f"    Top values: {cluster_data[feat].value_counts().head(3).to_dict()}")

### 4.2 Business Sense & Interpretation

In [ ]:
# Create comprehensive cluster interpretation

print("\n" + "="*100)
print("CLUSTER INTERPRETATION - DOES IT MAKE BUSINESS SENSE?")
print("="*100 + "\n")

cluster_interpretations = {}

for cluster_id in sorted(df['Cluster'].unique()):
    cluster_mask = df['Cluster'] == cluster_id
    cluster_data = df[cluster_mask]
    
    interpretation_text = f"""
CLUSTER {cluster_id}:
  • Size: {len(cluster_data)} customers ({100*len(cluster_data)/len(df):.1f}%)
  • Average features:
  """
    
    # Get numeric statistics
    numeric_stats = cluster_data[numeric_features].describe().T[['mean', 'std']]
    for feat in numeric_features[:5]:
        if feat in numeric_stats.index:
            interpretation_text += f"\n      - {feat}: {numeric_stats.loc[feat, 'mean']:.2f}"
    
    interpretation_text += f"""

  INTERPRETATION:
    Based on the important features identified by multiple models,
    this cluster can be characterized by [DETAILED DESCRIPTION BASED ON DATA].
    
    Business meaning: [INTERPRET IN BUSINESS CONTEXT]
    """
    
    cluster_interpretations[cluster_id] = interpretation_text
    print(interpretation_text)

### 4.3 Visualization - Cluster Profiles

In [ ]:
# Visualize cluster profiles using top features
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Cluster Profiles - Important Features', fontsize=16, fontweight='bold')

# Get top numeric features
top_numeric_features = [f for f in selected_features_names[:10] if f in numeric_features]

for idx, feat in enumerate(top_numeric_features[:4]):
    ax = axes[idx // 2, idx % 2]
    
    cluster_means = []
    cluster_ids = []
    
    for cluster_id in sorted(df['Cluster'].unique()):
        cluster_data = df[df['Cluster'] == cluster_id]
        if feat in cluster_data.columns:
            cluster_means.append(cluster_data[feat].mean())
            cluster_ids.append(f'C{cluster_id}')
    
    ax.bar(cluster_ids, cluster_means, color='steelblue')
    ax.set_title(f'Mean {feat} by Cluster')
    ax.set_ylabel('Mean Value')
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('results/cluster_profiles.png', dpi=300, bbox_inches='tight')
plt.show()

print("Saved: results/cluster_profiles.png")

## **Best Model Summary**

Surprisingly, the Optimized All-Features Model was the true winner, achieving the highest F1-Score of 91.75%. This happened because keeping every piece of data turned out to be smarter than simplifying it. While the smaller model threw away 'weak' clues, the full model used regularization to simply quiet them down without losing them entirely. This allowed it to capture hidden patterns where small details—useless on their own—became powerful when combined with others. By listening to every signal instead of cutting them out, the model gained the subtle insights needed to solve the most difficult cases

## **Clustering Analysis**

**Cluster 0:**

The main driver for this group is the number of inactive months, which has a strong positive score of 3.5. There are also small positive factors like the total amount spent (0.15) and the number of transactions (0.1). On the other hand, factors like credit limit (-0.07) and customer age (-0.06) have negative scores. This means that customers with lower credit limits and younger ages are more likely to land in this group.

Cluster 0 consists of customers with high inactivity periods but occasionally make transactions. They are characterized by long periods without activity combined with low credit limits, representing disengaged or dormant customers.

**Cluster 1:**

This group is defined by customers who have been with the bank for a long time (0.2) and hold multiple products (0.2). Other positive factors include having higher-tier cards (0.2) and moderate periods of inactivity (0.5). On the other hand, actual usage has a massive negative impact. The total amount of money spent (-20.0) and the number of transactions (-17.5) have very strong negative scores. This means that customers who spend a lot or use their cards frequently do not belong in this group.

Cluster 1 represents long-term, multi-product customers with low transaction activity. These customers maintain relationships through multiple products (cards, accounts) and longer tenure, but they transact infrequently and with small amounts - likely relationship banking customers who hold accounts but don't actively use them.

**Cluster 2:**

This group is driven mainly by heavy usage. The number of transactions (7.5) and the total amount spent (6.5) have very high positive scores, making them the strongest predictors. Higher credit limits (0.1) also play a small positive role. Conversely, factors like income level (-0.3), slowing transaction trends (-0.25), education level (-0.25), and the number of bank products (-0.2) have negative scores, meaning these traits are less common in this active group.

Cluster 2 contains highly active transactors with high spending. They are defined by extremely high transaction volumes and amounts, representing the most engaged and valuable customers from a transaction perspective, even though they may have fewer total banking products.

**Cluster 3:**

The most distinct feature of this group is that they are almost never inactive. The number of inactive months has a very strong negative score of -6.0, which pushes inactive people away from this cluster. On the positive side, these customers show an upward trend in activity. The frequency of their transactions (0.12) and the amount they spend (0.06) are increasing compared to the start of the year. Personal details like being divorced (0.1) or having a higher education level (0.04) are also small positive indicators for this group.

Cluster 3 represents customers showing growth in transaction behavior. They are characterized by increasing transaction trends (Q4 vs Q1) and minimal inactivity, suggesting customers whose engagement is growing or changing, possibly responding to life events (divorced/unknown marital status) or developing stronger banking relationships.

Looking at the story across all four groups, the most important plot point is transaction behavior. The data shows that how often customers swipe their cards and how much they spend are the main things that make them different from each other.

We see a clear "battle of opposites" between Cluster 1 and Cluster 2. One group focuses on having relationships and products but rarely spends, while the other spends heavily but holds fewer products. The dividing line in this story is inactivity. The number of dormant months acts as a wall that separates the engaged customers (Clusters 2 and 3) from the disengaged ones (Clusters 0 and 1). Finally, it turns out that personal details like education and income are just minor characters in this story—ultimately, what customers do with their accounts matters much more than who they are.
